# Couzin Zones:
Couzin defines 3 zones, each produces a different response on each of the fish's behaviour.

**Repulsion:** is the primordial. Let's take d = distance, if d<0.5 the fish will move away. Fish avoid collisions.

**Orientation:** if the neighbors are in a comfortable/ideal distance, the fish will calculate the medium of the neighbor's orientation and will move towards the same direction.

**Attraction:** if a fish is in a distance d<5.5, the fish will get close (because it's far neighbor still close to it). otherwise, if the neighbor is in a d>5.5, the fish will ignore it (it's out of its reach).

### For now we are just interested in the fish recognizing its relevant neighbors.

## Fisrt: 
We settle the interaction radius.

In [ ]:
''' Phase 1D: Couzin repulsion.
Objective: Detect nearby fish and avoid neighbors inside the repulsion zone (working on it)'''
import numpy as np
import mesa
from mesa.experimental.continuous_space import ContinuousSpaceAgent, ContinuousSpace
from mesa.visualization import SolaraViz, make_space_component
from matplotlib.markers import MarkerStyle

class GoldenShiners(ContinuousSpaceAgent):
    """ Creatin a Golden Shiner """
    def __init__(self, model, space, position=(0, 0), speed=1.0):
        super().__init__(space, model)
        self.position = np.array(position, dtype=float)
        self.speed = speed
     
        # Random initial direction:
        angle = self.model.rng.uniform(0, 2 * np.pi)
        self.direction = np.array([np.cos(angle), np.sin(angle)])

    def get_neighbors(self):
        neighbors = []
        for other_fish in self.model.agents:
            if other_fish is self:
                continue
            displacement = other_fish.position - self.position
            distance = np.linalg.vector_norm(displacement)
            if distance <= self.model.attraction_radius:
                neighbors.append((other_fish, displacement, distance))
        return neighbors

    #Trying out repulsion:
    def repulsion(self, neighbors):
        repel = np.zeros(2)
        for _, displacement, distance in neighbors: #each element of neighbors has:(other_fish, displacement, distance)
            if distance < self.model.repulsion_radius:
                if distance == 0: #Aqui tenemos que hacer debugg porque puede haber overlapping, o sea que la disancia sea cero
                    continue
                direction_neighbor = displacement/distance
                repel -= direction_neighbor
        magnitude = np.linalg.vector_norm(repel)
        if magnitude == 0:
            return None
        return repel/magnitude

    def attraction(self, neighbors):
        attract = np.zeros(2)
        for _, displacement, distance in neighbors:
            if self.model.orientation_radius <= distance <= self.model.attraction_radius:
                attract += displacement / distance
    
        magnitude = np.linalg.vector_norm(attract)
        return attract / magnitude if magnitude > 0 else None

#Hmmm two separate functions so one decides and anothers move to make sure they all move at same time??
    def decide(self):
        neighbors = self.get_neighbors()
        repel = self.repulsion(neighbors)
        self.next_direction = (repel if repel is not None else self.direction.copy())
        try:
            self.next_position = (self.position+self.next_direction * self.speed * self.model.dt)
        except self.next_direction * self.speed * self.model.dt >= model.repulsion_radius:
            # self.next_position = (self.position+self.next_direction...)
            coeff=(model.repulsion_radius-0.01)/(self.next_direction * self.speed * self.model.dt)
            self.next_position=(self.position+self.next_direction * self.speed * self.model.dt)*coeff
            

    def advance(self):
        self.direction = self.next_direction
        self.bounce(self.next_position)
        self.position = self.next_position

    #Making sure that the agent changes it's direction when it encounters a wall:
    def bounce(self, position):
        for axis,(minimum, maximum) in enumerate(self.model.bounds):
            if position[axis] < minimum:
                position[axis] = 2 * minimum - position[axis]
                self.direction[axis] *= -1
            elif position[axis] > maximum:
                position[axis] = 2 * maximum - position[axis]
                self.direction[axis] *= -1
                

In [4]:
class GoldenShinersModel(mesa.Model):
   """Phase 1D: repulsion."""
   def __init__(self, width=100, height=100, speed=1, n_fish= 20, seed=None):
    super().__init__(seed=seed)
    self.bounds = np.array([[0, width],[0, height] ])
    self.dt = 0.125

    # Fixed Couzin parameters
    self.repulsion_radius = 0.5
    self.orientation_radius = 3.0
    self.attraction_radius = 5.5

    # Create continuous space
    self.space = ContinuousSpace(self.bounds, torus=False, random=self.random)

    for _ in range(n_fish):
    # Create n fish
        position = self.rng.random(2) * np.array([width, height]) #chooses a position
        GoldenShiners(self, self.space, position, speed) #creates a fish in that position, in this case 20 fishes
        
    def step(self):
        self.agents.do("decide")
        self.agents.do("advance")



In [3]:
# Visualization
def agent_draw(agent):
    """Simple agent portrayal with arrow pointing in movement direction."""
    # Calculate angle from direction vector
    angle_rad = np.arctan2(agent.direction[1], agent.direction[0])
    angle_deg = np.degrees(angle_rad)
    
    # Create arrow marker pointing in agent's direction
    marker = MarkerStyle(marker='>')  # Arrow marker
    marker._transform = marker.get_transform().rotate_deg(angle_deg)
    return {"color": "blue", "size": 15, "marker": marker}

# Set up visualization
# model = GoldenShinersModel()


#To Visualiazite better let's reduce the space pf the fishes temporarily
model = GoldenShinersModel(width=20,height=20,n_fish=20,seed=1)
page = SolaraViz(model, components=[make_space_component(agent_portrayal=agent_draw)], name="Phase 1D: Repulsion"
)

page

C:\Users\fabio\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\mesa\mesa_logging.py:112: FutureWarning: The use of the `seed` keyword argument is deprecated, use `rng` instead. No functional changes.
  res = func(*args, **kwargs)
C:\Users\fabio\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\mesa\visualization\mpl_space_drawing.py:670: FutureWarning: Returning a dict from agent_portrayal is deprecated and will be removed in Mesa 4.0. Please return an AgentPortrayalStyle instance instead. For more information, refer to the migration guide: https://mesa.readthedocs.io/latest/migration_guide.html#defining-portrayal-components
  arguments = collect_agent_data(space, agent_portrayal, default_size=s_default)


Cannot show ipywidgets in text